# CEO-Request Challenge

## 1. Summary of Problem Statement

❓ **How could Olist improve its profit** ❓

### P&L Rules

#### Revenues  
**Sales fees:** Olist takes a **10% cut** on the product price (excl. freight) of each order delivered  
**Subscription fees:** Olist charges **80 BRL per month** per seller

#### Costs
_Estimated_ **reputation costs** of orders with bad reviews (<= 3 stars)  

💡 In the long term, bad customer experience has business implications: low repeat rate, immediate customer support cost, refunds or unfavorable word of mouth communication. We make an assumption about the monetary cost for each bad review:
```python
# review_score: cost(BRL)
{'1 star': 100
 '2 stars': 50
 '3 stars': 40
 '4 stars': 0
 '5 stars': 0}
```

**IT costs:** Olist's **total cumulated IT Costs** scale with the square root of the total number of sellers that have ever joined the platform, as well as with the square root of the total cumulated number of items that were ever sold.

$IT\_costs = \alpha * \sqrt{n\_sellers} + \beta * \sqrt{n\_items}$  
Olist's data team gave us the following values for these scaling parameters:
- $\alpha = 3157.27$
- $\beta = 978.23$

💡 Both the number of sellers to manage and the number of sales transaction are costly for IT systems.  
💡 Yet square roots reflect scale-effects: IT-system are often more efficient as they grow bigger.  
💡 Alpha > Beta means that Olist has a lower IT Cost with few sellers selling a lot of items rather than the opposite  
- with **1000 sellers** and a total of **100 items** sold, the total IT cost accumulates to 109,624 BRL
- with **100 sellers** and a total of **1000 items** sold, the total IT cost accumulates to 62,507 BRL

Finally, The IT department also told you that since the birth of the marketplace, cumulated IT costs have amounted to **500,000 BRL**.

### Key Findings, so far

- `wait_time` is the most significant factor behind low review scores.
- `wait_time` is made up of seller's `delay_to_carrier` + `carrier_delivery_time`.
- Because the carrier's delivery time is out of Olist's direct control, improving it is not a quick-win recommendation.
- On the contrary, a better selection of `sellers` can positively impact the `delay_to_carrier` and reduce the number of bad `review_scores` on Olist.
- Comments in the bad reviews showed that some were linked to the seller or to the product itself.

💡 We recommend you to start with the the guided seller analysis in part 2 below.

💪 But feel free to investigate other hypothesis instead with part 3.

## 2. Should Olist remove under-performing sellers from its marketplace? 🕵🏻
*(recommended)*

To analyze the impact of removing the worst sellers from Olist's marketplace, we will perform a **what-if analysis**

👉 **What would have happened if Olist had never accepted these sellers in the first place?**  

*(In practice, it's hard to know in advance who is a good seller, but let's start with this approach and iterate later).*

### 2.1 Data Preparation

Compute, for each `seller_id`, and cumulated since the beginning:
- the `revenues` the seller brings
- the `review_costs` associated with the seller's bad reviews
- the resulting `profits` (revenues - costs)

👉 Write down a step-by-step strategy to create the DataFrame you need.


⚠️ Don't start from scratch, update your existing package! 😉

Starting from the `Seller` class of your `olist` package:

Edit the `get_training_data` method so that the DataFrame it returns contains the fields:
- `revenues`: sum of subscription and sales fees revenues
- `cost_of_reviews`: sum of costs associated with bad reviews
- `profits`: `revenues` - `cost_of_reviews`

In [67]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

from olist.seller import Seller
from olist.order import Order
from olist.data import Olist
olist = Olist()
data = olist.get_data()
sellers = Seller().get_training_data()
orders = Order().get_training_data()



## Revenues

# Sales fees: Olist takes a 10% cut on the product price (excl. freight) of each order delivered
# Subscription fees: Olist charges 80 BRL per month per selle
from olist.seller import Seller
sellers_sales = Seller().get_sales()
active_dates = Seller().get_active_dates()


sales_fee= sellers_sales.sales *0.1
subscription_fee=active_dates.months_on_olist*80
revenues = sales_fee + subscription_fee

pd.DataFrame(revenues).rename(columns={0:'revenues'}).reset_index()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


/Users/lanwy/code/lanwyb/04-Decision-Science/01-Project-Setup/data-context-and-setup/olist/seller.py:100: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  df = orders_sellers.groupby('seller_id').agg({
/Users/lanwy/code/lanwyb/04-Decision-Science/01-Project-Setup/data-context-and-setup/olist/seller.py:100: FutureWarning: The provided callable <built-in function max> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df = orders_sellers.groupby('seller_id').agg({
/Users/lanwy/code/lanwyb/04-Decision-Science/01-Project-Setup/data-context-and-setup/olist/seller.py:100: FutureWarning: The provided callable <built-in function min> is currently using SeriesGroupBy.min. In a future version of 

,seller_id,revenues
0,0015a82c2db000af6aaaf3ae2ecb0532,348.500
1,001cca7ae9ae17fb1caed9dfb1094831,3868.003
2,001e6ad469a905060d959994f1b41e4f,25.000
3,002100f778ceb8431b7a1020ff7ab48f,683.450
4,003554e2dce176b5555353e4f3555ac8,12.000
...,...,...
3090,ffcfefa19b08742c5d315f2791395ee5,6.990
3091,ffdd9f82b9a447f6f8d4b91554cc7dd3,1650.120
3092,ffeee66ac5d5a62fe688b9d26f83f534,823.986
3093,fffd5413c0700ac820c7069d66d98c89,1946.230


In [ ]:
penalty = {'1 star': 100,
 '2 stars': 50,
 '3 stars': 40,
 '4 stars': 0,
 '5 stars': 0}


review_scores

review_scores = Seller().get_review_score() 
review_scores.set_index('seller_id')[['review_score']]
#np.array([x for i in review_scores])

review_scores.set_index('seller_id')[['review_score']].reset_index()

In [58]:
revenues

seller_id
0015a82c2db000af6aaaf3ae2ecb0532     348.500
001cca7ae9ae17fb1caed9dfb1094831    3868.003
001e6ad469a905060d959994f1b41e4f      25.000
002100f778ceb8431b7a1020ff7ab48f     683.450
003554e2dce176b5555353e4f3555ac8      12.000
                                      ...   
ffcfefa19b08742c5d315f2791395ee5       6.990
ffdd9f82b9a447f6f8d4b91554cc7dd3    1650.120
ffeee66ac5d5a62fe688b9d26f83f534     823.986
fffd5413c0700ac820c7069d66d98c89    1946.230
ffff564a4f9085cd26170f4732393726     622.630
Length: 3095, dtype: float64

### 2.2 What-if Analysis

👉 Time to perform the actual analysis, here are our steps:  

1️⃣ Write a function that will calculate IT costs based on two parameters: number of sellers and number of items.

2️⃣ Load the sellers data and sort them by decreasing profits (before IT costs).

3️⃣ Calculate profits:
   - Calculate the cumulative profits for each row.
   - Calculate the cumulative IT costs for each row using the function you defined before.
   - Calculate the cumulative net profit for each row.

4️⃣ Plot your results, and analyze them.

5️⃣ Determine the optimum number of sellers to keep, based on profits before and after IT costs. (Hint: look up `np.argmax`). What would have been the impact on:
   - Net profit after IT costs?
   - Net profit before IT costs
   - IT costs?
   - Total revenues?
   - Number of sellers?
   - Number of items sold?

6️⃣ How important were the IT costs in your analysis?

## 3. Investigate other Approaches 🕵️
*(optional)*

- Should Olist remove the worst performing products / categories from its marketplace entirely?
- Should Olist remove only consistently underperforming sellers, after a honeymoon period of a few months?
- Should Olist enforce sellers to include certain information on their product listings?
- Should Olist ask customers for purchase confirmation at certain times of day?
- Should Olist restrict seller/customer pairs between certain states to avoid delays?
- Should Olist acquire new sellers, with some cost assumptions to be suggested?
- ...


## Your turn!

🧺 Keep this notebook tidy! 

🗣 📊 You will present your insights to your favorite TA at the end of this `Communicate` unit 💪